In [0]:
SELECT
    h.provider_id,
    h.provider_name,
    h.provider_state,
    h.provider_city,
    h.hospital_type,
    h.hospital_ownership,
    h.total_discharges,
    h.avg_ctp_ratio,
    h.avg_revenue_gap_pct,
    h.underpayment_rate_pct,
    h.rcm_health_score,
    h.rcm_health_grade,
    ROUND(d.avg_denial_risk_score, 4)                  AS avg_denial_risk_score,
    d.high_risk_claims,
    d.medium_risk_claims,
    d.low_risk_claims
FROM rcm_platform.rcm_gold.hospital_scorecard h
LEFT JOIN (
    SELECT
        provider_id,
        ROUND(AVG(denial_risk_score), 4)               AS avg_denial_risk_score,
        SUM(CASE WHEN denial_risk_label = 'High Risk'
            THEN 1 ELSE 0 END)                         AS high_risk_claims,
        SUM(CASE WHEN denial_risk_label = 'Medium Risk'
            THEN 1 ELSE 0 END)                         AS medium_risk_claims,
        SUM(CASE WHEN denial_risk_label = 'Low Risk'
            THEN 1 ELSE 0 END)                         AS low_risk_claims
    FROM rcm_platform.rcm_gold.denial_risk_scores
    GROUP BY provider_id
) d ON h.provider_id = d.provider_id
ORDER BY h.total_discharges DESC